# Qwen2.5-1.5B Ablation Study — 完整实验流水线

在 A100 上依次完成：

| Phase | 内容 | 预计耗时 |
|-------|------|----------|
| **P0** | 环境 + Drive + 数据同步 | ~5 min |
| **P1** | Group B 扩充评测（SFT / SFT+DPO，n=200）| ~1.5h |
| **P2** | 错误分类（L3）+ Targeted DPO 数据生成（L4）| ~30 min |
| **P3** | Group A 训练（LoRA baseline）+ 评测 | ~4h |
| **P4** | Group D 训练（Error-Type-Targeted DPO）+ 评测 | ~3.5h |
| **P5** | 汇总 + 同步 | ~2 min |

**断点续跑**：每个 Phase 开始前检查 checkpoint / 结果文件，已完成则跳过。

**前提**：
- Google Drive `Qwen-Reasoning/` 下有 `outputs/sft_merged/` 和 `outputs/merged/`
- Colab Secrets 中有 `HF_TOKEN` 和 `DASHSCOPE_API_KEY`（P2 需要）
- A100 40GB 或 80GB

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P0: 环境准备 + Drive 挂载 + 代码克隆 + 数据同步
# 预计耗时：~5 min
# ═══════════════════════════════════════════════════════════════════

# ── 0.1 安装依赖 ──────────────────────────────────────────────────
!pip install -q -U pip
!pip install -q "unsloth>=2025.1.0" "trl>=0.14.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" \
    "transformers>=4.49.0" "datasets>=3.2.0" "accelerate>=1.2.0" \
    "pyyaml>=6.0.2" "safetensors" "tqdm" "scipy" "matplotlib" "seaborn"

import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB')

In [ ]:
# ── 0.2 挂载 Drive + 加载 Secrets ─────────────────────────────────
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/Qwen-Reasoning'

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN: OK')
except Exception:
    print('HF_TOKEN: missing（HuggingFace 下载可能受限）')

try:
    os.environ['DASHSCOPE_API_KEY'] = userdata.get('DASHSCOPE_API_KEY')
    print('DASHSCOPE_API_KEY: OK（P2 错误分类可用）')
except Exception:
    print('DASHSCOPE_API_KEY: missing（P2 将跳过，需本地运行 L3+L4）')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# ── 0.3 克隆代码 + 软链 outputs ───────────────────────────────────
import subprocess, shutil

PROJECT_DIR = '/content/6000Q-QwenMiniReason'
REPO_URL = 'https://github.com/yukiiii0730/6000Q-QwenMiniReason.git'

if not os.path.isdir(f'{PROJECT_DIR}/.git'):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=False)

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, f'{PROJECT_DIR}/eval')
sys.path.insert(0, f'{PROJECT_DIR}/scripts')

# outputs/ 软链到 Drive（训练产物实时持久化）
out_dir = f'{PROJECT_DIR}/outputs'
if not os.path.islink(out_dir):
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.symlink(f'{DRIVE_DIR}/outputs', out_dir)
print(f'outputs/ -> {os.readlink(out_dir)}')

# 同步 Drive 的 logs 到项目目录
os.makedirs('logs', exist_ok=True)
subprocess.run(['rsync', '-av', f'{DRIVE_DIR}/logs/', 'logs/'], capture_output=True)

In [ ]:
# ── 0.4 同步数据 + 检查模型 ───────────────────────────────────────
import json

# 同步训练数据
os.makedirs('data/processed', exist_ok=True)
subprocess.run(['rsync', '-av', f'{DRIVE_DIR}/data/processed/', 'data/processed/'],
               capture_output=True)

# 检查模型文件
for label, path in [
    ('Group B SFT', 'outputs/sft_merged'),
    ('Group B DPO', 'outputs/merged'),
    ('SFT adapter', 'outputs/sft'),
    ('DPO adapter', 'outputs/dpo'),
]:
    cfg = os.path.isfile(f'{path}/config.json')
    adp = os.path.isfile(f'{path}/adapter_config.json')
    ok = cfg or adp
    print(f'  {label:20s}: {"OK" if ok else "MISSING"} ({path})')

# 检查训练数据
for f in ['sft_train.json', 'dpo_train.json', 'dpo_teacher_round_1.json']:
    fp = f'data/processed/{f}'
    if os.path.isfile(fp):
        size = os.path.getsize(fp) / 1024 / 1024
        print(f'  {f:35s}: {size:.1f} MB')
    else:
        print(f'  {f:35s}: MISSING')

# A100 batch size 建议
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 4 if vram > 70 else 2
print(f'\nA100 batch_size={BATCH_SIZE}, grad_accum=8, effective_BS={BATCH_SIZE * 8}')

---
## P1: Group B 扩充评测（n=200）

对已有的 Group B 模型（SFT / SFT+DPO）进行 n=200 的扩充评测，产出：
- GSM8K + MATH-500 + BBH-27 的评测 JSON
- GSM8K SFT badcase 文件（供 P2 错误分类用）

预计耗时：~1.5h（3 benchmark × 2 模型 × ~15 min）

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P1: Group B 扩充评测
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, json, glob, shutil

os.chdir(PROJECT_DIR)
BIT = ['--load_in_4bit']
GSM8K_N, MATH_N, BBH_PER = '200', '200', '30'
SEP = '=' * 60


def eval_model(label, model_path, tag):
    """对一个模型跑 GSM8K + MATH + BBH，含 badcase 输出。"""
    print(f'\n{SEP}')
    print(f'  评测 {label} -> {model_path}')
    print(f'{SEP}')

    # GSM8K（必跑，产出 badcase 给 P2 用）
    gsm_out = f'logs/gsm8k_{tag}.json'
    gsm_bc = f'logs/gsm8k_{tag}_badcases.jsonl'
    if os.path.isfile(gsm_out):
        with open(gsm_out) as f:
            r = json.load(f)
        print(f'  GSM8K: already done ({r.get("accuracy", 0):.1%})')
    else:
        print(f'  GSM8K n={GSM8K_N} ...')
        subprocess.run([
            'python3', 'eval/gsm8k_eval.py',
            '--model_path', model_path,
            '--max_samples', GSM8K_N,
            '--sampling_mode', 'stratified',
            '--output', gsm_out,
            '--badcase_output', gsm_bc,
        ] + BIT, check=True)

    # MATH-500
    math_out = f'logs/math_{tag}.json'
    if os.path.isfile(math_out):
        with open(math_out) as f:
            r = json.load(f)
        print(f'  MATH:  already done ({r.get("accuracy", 0):.1%})')
    else:
        print(f'  MATH-500 n={MATH_N} ...')
        subprocess.run([
            'python3', 'eval/math_eval.py',
            '--model_path', model_path,
            '--max_samples', MATH_N,
            '--output', math_out,
        ] + BIT, check=True)

    # BBH-27
    bbh_out = f'logs/bbh_{tag}.json'
    if os.path.isfile(bbh_out):
        print(f'  BBH:   already done')
    else:
        print(f'  BBH-27 ({BBH_PER}/task x 27) ...')
        subprocess.run([
            'python3', 'eval/bbh_full_eval.py',
            '--mode', 'local',
            '--model_path', model_path,
            '--max_samples', BBH_PER,
            '--output_dir', f'logs/bbh_{tag}_eval',
        ] + BIT, check=True)
        summaries = glob.glob(f'logs/bbh_{tag}_eval/*summary*')
        if summaries:
            shutil.copy2(summaries[0], bbh_out)

    # 打印结果
    for name, path in [('GSM8K', gsm_out), ('MATH', math_out), ('BBH', bbh_out)]:
        if os.path.isfile(path):
            d = json.load(open(path))
            acc = d.get('accuracy') or d.get('macro_avg_accuracy', 0)
            print(f'  {name}: {acc:.2%}')


# ── Group B SFT ──────────────────────────────────────────────────
eval_model('Group B SFT', 'outputs/sft_merged', 'sft')

# ── Group B SFT+DPO ─────────────────────────────────────────────
eval_model('Group B SFT+DPO', 'outputs/merged', 'result')

# 同步 logs 到 Drive
subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
print('\nP1 完成，logs 已同步到 Drive')

---
## P2: 错误分类（L3）+ Targeted DPO 数据生成（L4）

用 qwen-flash 对 SFT badcase 做 5 类错误分类，再用 qwen3-235b 生成类型专属 DPO 数据。

**需要 DASHSCOPE_API_KEY**。如果没有，跳过此 Phase，在本地运行：
```bash
bash run_local_pipeline.sh    # L3+L4
```
然后把 `data/processed/dpo_targeted_round1.json` 上传到 Drive。

预计耗时：~30 min（L3 约 5 min，L4 约 25 min）

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P2: 错误分类 + Targeted DPO 数据生成
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, json

os.chdir(PROJECT_DIR)

# 检查是否已有 targeted DPO 数据（从 Drive 同步的或之前生成的）
TARGETED_DPO = 'data/processed/dpo_targeted_round1.json'
if os.path.isfile(TARGETED_DPO):
    data = json.load(open(TARGETED_DPO))
    print(f'dpo_targeted_round1.json 已存在 ({len(data)} 条)，跳过 P2')
else:
    api_key = os.environ.get('DASHSCOPE_API_KEY', '')
    if not api_key:
        print('DASHSCOPE_API_KEY 未设置，跳过 P2')
        print('请在本地运行: bash run_local_pipeline.sh')
        print('然后上传 data/processed/dpo_targeted_round1.json 到 Drive')
    else:
        # ── L3: 错误分类 ───────────────────────────────────────────
        BADCASE = 'logs/gsm8k_sft_badcases.jsonl'
        if not os.path.isfile(BADCASE):
            print(f'找不到 {BADCASE}，请先完成 P1 的 SFT 评测')
        else:
            print(f'L3: 错误分类 ({BADCASE}) ...')
            subprocess.run([
                'python3', 'scripts/classify_errors.py',
                '--badcase_jsonl', BADCASE,
                '--output_dir', 'results/errors/sft',
                '--workers', '4',
            ], check=True)
            print('L3 完成')

            # ── L4: 生成 Targeted DPO 数据 ─────────────────────────
            print(f'L4: 生成 Targeted DPO 数据 ...')
            subprocess.run([
                'python3', 'scripts/build_targeted_dpo.py',
                '--by_type_dir', 'results/errors/sft/by_type',
                '--per_type_n', '200',
                '--tag', 'round1',
                '--output_dir', 'data/processed',
                '--workers', '4',
            ], check=True)

            if os.path.isfile(TARGETED_DPO):
                data = json.load(open(TARGETED_DPO))
                print(f'L4 完成: {len(data)} 条 targeted DPO 数据')
                # 同步到 Drive
                subprocess.run(['rsync', '-av', 'data/processed/',
                                f'{DRIVE_DIR}/data/processed/'], capture_output=True)
                subprocess.run(['rsync', '-av', 'results/',
                                f'{DRIVE_DIR}/results/'], capture_output=True)
                print('数据已同步到 Drive')

---
## P3: Group A 训练（LoRA + 单段混合 SFT + Standard DPO）

作为经典 baseline，与 Group B（DoRA + 五段课程）对比。

| 配置 | Group A | Group B |
|------|---------|--------|
| PEFT | LoRA | DoRA |
| SFT 数据 | 单段混合 15k | 五段课程 38k |
| DPO 数据 | distilabel 5k | distilabel 5k |
| DPO loss | sigmoid | sigmoid |

预计耗时：~4h（SFT ~2.5h + DPO ~1.5h）

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P3: Group A 训练
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, yaml, glob

os.chdir(PROJECT_DIR)

G_A_SFT = 'outputs/group_a/sft'
G_A_SFT_MERGED = 'outputs/group_a/sft_merged'
G_A_DPO = 'outputs/group_a/dpo'
G_A_MERGED = 'outputs/group_a/merged'


# ── P3.1: Group A SFT（LoRA + 单段混合）──────────────────────────
if os.path.isfile(f'{G_A_SFT_MERGED}/config.json'):
    print('Group A sft_merged 已存在，跳过 SFT 训练')
else:
    print('Group A SFT 训练（LoRA + 单段混合，~2.5h on A100）...')

    # 生成 Group A 专用 SFT 配置
    sft_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'output_dir': G_A_SFT,
        'max_seq_length': 2048,
        'load_in_4bit': True,
        'seed': 42,
        'stages': [{
            'name': 'stage_mixed',
            'dataset': {
                'path': 'data/processed/sft_train.json',
                'max_samples': 15000,
                'field_map': {'prompt': 'input', 'response': 'output'},
            },
            'train': {
                'max_steps': 900,
                'learning_rate': 3e-5,
                'warmup_steps': 90,
            },
        }],
        'lora': {
            'use_dora': False,  # 关键区别：LoRA 而非 DoRA
            'r': 16, 'alpha': 32, 'dropout': 0.0,
            'target_modules': ['q_proj','k_proj','v_proj','o_proj',
                                'gate_proj','up_proj','down_proj'],
        },
        'train': {
            'per_device_train_batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': 8,
            'warmup_steps': 90,
            'max_steps': 900,
            'learning_rate': 3e-5,
            'logging_steps': 10,
            'save_steps': 300,
            'weight_decay': 0.01,
            'lr_scheduler_type': 'cosine',
            'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True,
            'dataloader_num_workers': 4,
            'packing': True,
        },
        'dataset_path': 'data/processed/sft_train.json',
    }
    os.makedirs('config', exist_ok=True)
    with open('config/sft_config_group_a.yaml', 'w') as f:
        yaml.dump(sft_cfg, f, allow_unicode=True)

    # sft_train.py 自动检测最新 checkpoint 并续训
    subprocess.run([
        'python3', 'scripts/sft_train.py',
        '--config', 'config/sft_config_group_a.yaml',
    ], check=True)

    # 合并 LoRA
    print('合并 Group A SFT adapter ...')
    os.makedirs(G_A_SFT_MERGED, exist_ok=True)
    subprocess.run([
        'python3', 'scripts/merge_lora.py',
        '--adapter_path', G_A_SFT,
        '--output_path', G_A_SFT_MERGED,
    ], check=True)

# 同步到 Drive
subprocess.run(['rsync', '-av', 'outputs/group_a/', f'{DRIVE_DIR}/outputs/group_a/'],
               capture_output=True)
print('Group A SFT 完成')

In [ ]:
# ── P3.2: Group A SFT 评测 ──────────────────────────────────────
import subprocess, os, json

os.chdir(PROJECT_DIR)

G_A_SFT_MERGED = 'outputs/group_a/sft_merged'

print('Group A SFT 评测 ...')
for ds, script, n in [
    ('gsm8k', 'eval/gsm8k_eval.py', '200'),
    ('math',  'eval/math_eval.py',  '200'),
]:
    out = f'logs/group_a_sft_{ds}.json'
    if os.path.isfile(out):
        print(f'  {ds}: already done')
    else:
        cmd = ['python3', script, '--model_path', G_A_SFT_MERGED,
               '--max_samples', n, '--output', out]
        if ds == 'gsm8k':
            cmd += ['--sampling_mode', 'stratified',
                    '--badcase_output', 'logs/group_a_sft_gsm8k_badcases.jsonl']
        subprocess.run(cmd + BIT, check=True)

bbh_out = 'logs/group_a_sft_bbh.json'
if not os.path.isfile(bbh_out):
    subprocess.run([
        'python3', 'eval/bbh_full_eval.py', '--mode', 'local',
        '--model_path', G_A_SFT_MERGED, '--max_samples', BBH_PER,
        '--output_dir', 'logs/group_a_sft_bbh_eval',
    ] + BIT, check=True)
    sums = glob.glob('logs/group_a_sft_bbh_eval/*summary*')
    if sums: shutil.copy2(sums[0], bbh_out)

subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
print('Group A SFT 评测完成')

In [ ]:
# ── P3.3: Group A DPO 训练 ──────────────────────────────────────
import subprocess, os, yaml, glob

os.chdir(PROJECT_DIR)

G_A_SFT_MERGED = 'outputs/group_a/sft_merged'
G_A_DPO = 'outputs/group_a/dpo'
G_A_MERGED = 'outputs/group_a/merged'

if os.path.isfile(f'{G_A_MERGED}/config.json'):
    print('Group A merged 已存在，跳过 DPO')
else:
    print('Group A DPO 训练（Standard sigmoid，~1.5h on A100）...')

    dpo_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'base_adapter_path': G_A_SFT_MERGED,
        'output_dir': G_A_DPO,
        'max_seq_length': 2048,
        'load_in_4bit': True,
        'seed': 42,
        'beta': 0.1,
        'loss_type': 'sigmoid',
        'dataset': {
            'name': 'argilla/distilabel-math-preference-dpo',
            'split': 'train',
            'max_samples': 5000,
        },
        'lora': {
            'use_dora': False,  # Group A 保持 LoRA
            'r': 16, 'alpha': 32, 'dropout': 0.0,
            'target_modules': ['q_proj','k_proj','v_proj','o_proj',
                                'gate_proj','up_proj','down_proj'],
        },
        'train': {
            'per_device_train_batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': 8,
            'warmup_steps': 50,
            'max_steps': 600,
            'learning_rate': 1e-5,
            'logging_steps': 10,
            'save_steps': 100,
            'weight_decay': 0.0,
            'lr_scheduler_type': 'cosine',
            'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True,
            'dataloader_num_workers': 4,
        },
        'dataset_path': 'data/processed/dpo_train.json',
    }
    with open('config/dpo_config_group_a.yaml', 'w') as f:
        yaml.dump(dpo_cfg, f, allow_unicode=True)

    # dpo_train.py 自动检测最新 checkpoint 并续训
    subprocess.run([
        'python3', 'scripts/dpo_train.py',
        '--config', 'config/dpo_config_group_a.yaml',
    ], check=True)

    # 合并
    print('合并 Group A DPO adapter ...')
    os.makedirs(G_A_MERGED, exist_ok=True)
    subprocess.run([
        'python3', 'scripts/merge_lora.py',
        '--adapter_path', G_A_DPO,
        '--output_path', G_A_MERGED,
    ], check=True)

subprocess.run(['rsync', '-av', 'outputs/group_a/', f'{DRIVE_DIR}/outputs/group_a/'],
               capture_output=True)
print('Group A DPO 完成')

In [ ]:
# ── P3.4: Group A 最终评测 ──────────────────────────────────────
import subprocess, os, json, glob, shutil

os.chdir(PROJECT_DIR)
G_A_MERGED = 'outputs/group_a/merged'

print('Group A 最终评测 ...')
for ds, script, n in [
    ('gsm8k', 'eval/gsm8k_eval.py', '200'),
    ('math',  'eval/math_eval.py',  '200'),
]:
    out = f'logs/group_a_{ds}.json'
    if os.path.isfile(out):
        print(f'  {ds}: already done')
    else:
        cmd = ['python3', script, '--model_path', G_A_MERGED,
               '--max_samples', n, '--output', out]
        if ds == 'gsm8k':
            cmd += ['--sampling_mode', 'stratified']
        subprocess.run(cmd + BIT, check=True)

bbh_out = 'logs/group_a_bbh.json'
if not os.path.isfile(bbh_out):
    subprocess.run([
        'python3', 'eval/bbh_full_eval.py', '--mode', 'local',
        '--model_path', G_A_MERGED, '--max_samples', BBH_PER,
        '--output_dir', 'logs/group_a_bbh_eval',
    ] + BIT, check=True)
    sums = glob.glob('logs/group_a_bbh_eval/*summary*')
    if sums: shutil.copy2(sums[0], bbh_out)

subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
print('Group A 完成')

---
## P4: Group D 训练（Error-Type-Targeted DPO）

**创新核心**：用 P2 生成的 targeted DPO 数据 + error_type 加权训练。

**前提**：`data/processed/dpo_targeted_round1.json` 已存在（P2 生成或本地上传）。

| 配置 | Group D | Group B |
|------|---------|--------|
| SFT | DoRA + 五段课程（共用）| DoRA + 五段课程 |
| DPO 数据 | **Targeted（类型专属）** | distilabel 通用 |
| DPO loss | sigmoid + **error_type 加权** | sigmoid |

预计耗时：~1.5h

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P4: Group D 训练
# ═══════════════════════════════════════════════════════════════════
import subprocess, os, yaml, json, glob, shutil

os.chdir(PROJECT_DIR)

G_D_DPO = 'outputs/group_d/dpo'
G_D_MERGED = 'outputs/group_d/merged'
SFT_MERGED = 'outputs/sft_merged'  # Group B 的 SFT（共用基座）
TARGETED_DPO = 'data/processed/dpo_targeted_round1.json'

# 前提检查
if not os.path.isfile(f'{SFT_MERGED}/config.json'):
    print(f'SFT 基座 {SFT_MERGED} 不存在，请先完成 P1')
elif not os.path.isfile(TARGETED_DPO):
    print(f'{TARGETED_DPO} 不存在')
    print('请先完成 P2，或在本地运行 bash run_local_pipeline.sh 后上传到 Drive')
elif os.path.isfile(f'{G_D_MERGED}/config.json'):
    print('Group D merged 已存在，跳过训练')
else:
    data = json.load(open(TARGETED_DPO))
    print(f'Targeted DPO 数据: {len(data)} 条')
    if 'error_type' in data[0]:
        from collections import Counter
        types = Counter(d.get('error_type', '?') for d in data)
        for t, n in types.most_common():
            print(f'  {t}: {n}')

    print('\nGroup D DPO 训练（Error-Type-Targeted，~1.5h on A100）...')

    dpo_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'base_adapter_path': SFT_MERGED,
        'output_dir': G_D_DPO,
        'max_seq_length': 2048,
        'load_in_4bit': True,
        'seed': 42,
        'beta': 0.1,
        'loss_type': 'sigmoid',
        # 创新核心：按错误类型加权
        'error_type_weights': {
            'arithmetic': 1,
            'reasoning_skip': 2,
            'setup_error': 3,
            'unit_or_format': 1,
            'extraction_error': 1,
        },
        'dataset': {
            'name': 'local',
            'split': 'train',
            'max_samples': -1,
        },
        'lora': {
            'use_dora': True,  # 与 Group B 一致
            'r': 16, 'alpha': 32, 'dropout': 0.0,
            'target_modules': ['q_proj','k_proj','v_proj','o_proj',
                                'gate_proj','up_proj','down_proj'],
        },
        'train': {
            'per_device_train_batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': 8,
            'warmup_steps': 50,
            'max_steps': 600,
            'learning_rate': 1e-5,
            'logging_steps': 10,
            'save_steps': 100,
            'weight_decay': 0.0,
            'lr_scheduler_type': 'cosine',
            'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True,
            'dataloader_num_workers': 4,
        },
        'dataset_path': TARGETED_DPO,
    }
    os.makedirs('config', exist_ok=True)
    with open('config/dpo_config_group_d.yaml', 'w') as f:
        yaml.dump(dpo_cfg, f, allow_unicode=True)

    # dpo_train.py 自动检测最新 checkpoint 并续训
    subprocess.run([
        'python3', 'scripts/dpo_train.py',
        '--config', 'config/dpo_config_group_d.yaml',
    ], check=True)

    print('合并 Group D adapter ...')
    os.makedirs(G_D_MERGED, exist_ok=True)
    subprocess.run([
        'python3', 'scripts/merge_lora.py',
        '--adapter_path', G_D_DPO,
        '--output_path', G_D_MERGED,
    ], check=True)

    subprocess.run(['rsync', '-av', 'outputs/group_d/', f'{DRIVE_DIR}/outputs/group_d/'],
                   capture_output=True)
    print('Group D 训练完成')

In [ ]:
# ── P4.2: Group D 评测 ──────────────────────────────────────────
import subprocess, os, json, glob, shutil

os.chdir(PROJECT_DIR)
G_D_MERGED = 'outputs/group_d/merged'

if not os.path.isfile(f'{G_D_MERGED}/config.json'):
    print('Group D 模型不存在，跳过评测')
else:
    print('Group D 评测 ...')
    for ds, script, n in [
        ('gsm8k', 'eval/gsm8k_eval.py', '200'),
        ('math',  'eval/math_eval.py',  '200'),
    ]:
        out = f'logs/group_d_{ds}.json'
        if os.path.isfile(out):
            print(f'  {ds}: already done')
        else:
            cmd = ['python3', script, '--model_path', G_D_MERGED,
                   '--max_samples', n, '--output', out]
            if ds == 'gsm8k':
                cmd += ['--sampling_mode', 'stratified',
                        '--badcase_output', 'logs/group_d_gsm8k_badcases.jsonl']
            subprocess.run(cmd + BIT, check=True)

    bbh_out = 'logs/group_d_bbh.json'
    if not os.path.isfile(bbh_out):
        subprocess.run([
            'python3', 'eval/bbh_full_eval.py', '--mode', 'local',
            '--model_path', G_D_MERGED, '--max_samples', BBH_PER,
            '--output_dir', 'logs/group_d_bbh_eval',
        ] + BIT, check=True)
        sums = glob.glob('logs/group_d_bbh_eval/*summary*')
        if sums: shutil.copy2(sums[0], bbh_out)

    subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
    print('Group D 完成')

---
## P5: 汇总 + 同步

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P5: 汇总所有实验结果
# ═══════════════════════════════════════════════════════════════════
import json, os, subprocess
from pathlib import Path

os.chdir(PROJECT_DIR)

def load_acc(path):
    if not Path(path).exists():
        return None
    d = json.loads(Path(path).read_text())
    return d.get('accuracy') or d.get('macro_avg_accuracy')

groups = [
    ('A', 'LoRA + 单段SFT + Std DPO',
     'logs/group_a_gsm8k.json', 'logs/group_a_math.json', 'logs/group_a_bbh.json'),
    ('B-SFT', 'DoRA + 五段课程 (SFT only)',
     'logs/gsm8k_sft.json', 'logs/math_sft.json', 'logs/bbh_sft.json'),
    ('B', 'DoRA + 五段课程 + Std DPO',
     'logs/gsm8k_result.json', 'logs/math_result.json', 'logs/bbh_result.json'),
    ('D', 'DoRA + 五段课程 + Targeted DPO',
     'logs/group_d_gsm8k.json', 'logs/group_d_math.json', 'logs/group_d_bbh.json'),
]

print('=' * 70)
print(f'{"Group":8} {"Description":40} {"GSM8K":>8} {"MATH":>8} {"BBH":>8}')
print('-' * 70)

rows = []
for g, desc, gsm, math, bbh in groups:
    ga, ma, ba = load_acc(gsm), load_acc(math), load_acc(bbh)
    g_str = f'{ga*100:.1f}%' if ga else '  N/A'
    m_str = f'{ma*100:.1f}%' if ma else '  N/A'
    b_str = f'{ba*100:.1f}%' if ba else '  N/A'
    print(f'{g:8} {desc:40} {g_str:>8} {m_str:>8} {b_str:>8}')
    rows.append({'group': g, 'desc': desc,
                 'gsm8k': ga, 'math': ma, 'bbh': ba})

print('=' * 70)

# 保存汇总
os.makedirs('results/ablation', exist_ok=True)
with open('results/ablation/summary_table.json', 'w') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

# 更新对比表
subprocess.run(['python3', 'eval/compare_table.py'], check=False)

# 最终同步
subprocess.run(['rsync', '-av', 'logs/', f'{DRIVE_DIR}/logs/'], capture_output=True)
subprocess.run(['rsync', '-av', 'results/', f'{DRIVE_DIR}/results/'], capture_output=True)
subprocess.run(['rsync', '-av', 'outputs/', f'{DRIVE_DIR}/outputs/'], capture_output=True)

print('\n全部完成！结果已同步到 Drive。')
print('下一步：下载 logs/ 到本地，运行 bash run_local_pipeline.sh --l5-only --l6-only')